# 🚇 MTA Subway Screensaver: Data Pipeline

This notebook turns public MTA data into everything the screensaver needs: the map, where every station is on it, and where every train is right now.

**How to run:** *Run → Run All Cells* (about 30 seconds, needs internet). Every section depends only on the sections above it.

### Data sources

| Source | What it is | Updates | Used in |
|---|---|---|---|
| **Static GTFS** `gtfs_subway.zip` | The schedule: stations, routes, colors, track shapes, timetables | A few times a year | §2, §3 |
| **Official subway diagram** (PDF) | The MTA map, as vector artwork | When the map is redesigned | §4 |
| **GTFS-Realtime** (8 feeds) | Live predictions for every trip | About every 30 s | §5, §6 |

### Project files

| Path | What it holds |
|---|---|
| `subway_pipeline.ipynb` | This notebook |
| `mtamap/` | Python package that reads the MTA map PDF and finds each station on it (§4) |
| `data/` | Downloads and generated images; safe to delete, everything is re-created |
| `export/` | What the Swift screensaver reads: `map_black.pdf` + `network.json` (§8) |
| `live_demo.ipynb`, `fast_demo.py` | Animated previews (in the notebook, and in a native window) |
| `archive/` | Earlier versions of this notebook |

### Glossary

| Term | Meaning | Example |
|---|---|---|
| **station** | A GTFS *parent station*: one location on one line | `L27` New Lots Av |
| **platform** (stop) | One direction of a station; the live feed always uses these | `L27S` (southbound) |
| **complex** | Several GTFS stations joined by free transfers | Times Sq = `127`, `725`, `902`, `R16`, `A27` |
| **route** | A service letter/number | `L`, `6`, `6X` (diamond 6) |
| **trip** | One run of one train | `127850_L..S` |
| **pattern** | The ordered list of platforms a trip stops at; many trips share one | L to Canarsie, all stops |
| **segment** | Track between two consecutive platforms in a pattern | `L27S → L28S` |
| **hop** | Same idea on the map: one route between two stations | the L from `L27` to `L28` |

### Roadmap

1. Setup ✅  2. Schedule ✅  3. Track geometry ✅  4. Official map ✅  5. Live feeds ✅  6. Placing trains ✅  7. Focused views ✅  8. Export ✅
Next (see §9): the Swift screensaver · smooth motion · next train at your station · service status

Appendix A keeps a generated map (shareable, unlike the official one). Appendix B has the earlier exercises and checks.

---
## 1. Setup

All imports, then **every setting in one place**. Change a value here and re-run the notebook; nothing else hard-codes these numbers.

In [ ]:
import time
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import requests
from google.transit import gtfs_realtime_pb2
from pyproj import Transformer
from shapely.geometry import LineString, Point
from shapely.ops import substring

import mtamap

# Shapely/GEOS prints a harmless "invalid value encountered" RuntimeWarning on a few MTA track shapes.
# Checked: every result is still a valid number, so we hide just this one warning.
warnings.filterwarnings("ignore", message="invalid value encountered", category=RuntimeWarning)

In [ ]:
# ---- Files and URLs ----
DATA = Path("data")  # downloads + generated images (safe to delete; re-created on the next run)
DATA.mkdir(exist_ok=True)

STATIC_URL = "https://rrgtfsfeeds.s3.amazonaws.com/gtfs_subway.zip"  # MTA static GTFS (the schedule)
STATIC_ZIP = DATA / "gtfs_subway.zip"
STATIC_MAX_AGE_DAYS = 7  # re-download the schedule when our copy is older than this

MAP_URL = "https://www.mta.info/map/5256"  # the official subway diagram (a PDF)
MAP_PDF = DATA / "subway_diagram.pdf"

STATIONS_URL = "https://data.ny.gov/api/views/39hk-dx4f/rows.csv?accessType=DOWNLOAD"  # MTA station list (open data)
STATIONS_CSV = DATA / "mta_stations.csv"  # used for official direction labels ("Uptown & The Bronx", …)

EXPORT_DIR = Path("export")  # files the Swift screensaver reads (§8); re-created every time §8 runs

FEED_URL = "https://api-endpoint.mta.info/Dataservice/mtagtfsfeeds/nyct%2F{}"  # {} = feed name; no API key needed
FEEDS = ["gtfs", "gtfs-ace", "gtfs-bdfm", "gtfs-g", "gtfs-jz", "gtfs-nqrw", "gtfs-l", "gtfs-si"]  # "gtfs" = 1-7 & S

# ---- Track geometry (§3) ----
SIMPLIFY_TOL_FT = 25  # drop track-shape detail smaller than this (one screen pixel ≈ 72 ft; see Appendix B)

# ---- Placing trains (§6) ----
STALE_AFTER_S = 5 * 60  # a trip whose next predicted time is this far in the past is a leftover: ignore it
PARK_WINDOW_S = 10 * 60  # show not-yet-departed trains (dimmed, at their terminal) only if they leave this soon
DEFAULT_HOP_S = 90  # travel time to assume between two stops when the schedule has none (the median hop is ~90 s)

# ---- Focused views (§7) ----
# Map coordinates: the whole map is 1656 units wide × 1978 tall (1 unit = 1 pt on the printed map).
SCREEN_ASPECT = 3024 / 1964  # MacBook screen width ÷ height; focused views are framed to fill it exactly
FOCUS_PAD = 30  # breathing room (map units) around the chosen stations
FOCUS_MIN_WIDTH = 220  # never zoom in tighter than this many units across, so a one-station view stays readable
SAME_NAME_RADIUS = 150  # stations sharing a name count as one place only if they're this close on the map
TRAIN_DOT_UNITS = 6.5  # train dot diameter in map units in focused views (a station circle is 8.2)

# name → stations to keep in frame. Use exact station names or GTFS station IDs (find_stations("…") lists them).
# A station brings the rest of its complex with it, e.g. "Times Sq-42 St" includes the Port Authority A/C/E.
FOCUS_VIEWS = {
    "Midtown": ["Times Sq-42 St", "Grand Central-42 St", "34 St-Herald Sq", "34 St-Penn Station",
                "Lexington Av/59 St", "59 St-Columbus Circle"],
    "Lower Manhattan": ["WTC Cortlandt", "A38",  # A38 = Fulton St ("Fulton St" alone is ambiguous: one is in Brooklyn)
                        "Bowling Green", "Brooklyn Bridge-City Hall", "Chambers St"],
    "Downtown Brooklyn": ["Jay St-MetroTech", "Borough Hall", "R30",  # R30 = DeKalb Av (another is on the L)
                          "Atlantic Av-Barclays Ctr", "Hoyt-Schermerhorn Sts"],
    "Union Sq": ["14 St-Union Sq"],
    "Long Island City": ["Court Sq", "Queens Plaza", "Queensboro Plaza", "Hunters Point Av"],
    # "Home": ["…"],  # add your own station(s) here
}

# Map matching fixes (§4): stations the automatic matching still places on the wrong circle group.
# Each entry is station ID → a map point (x, y) on its correct circles; the station takes the group nearest it.
# To find a point: look at the station on the §7 gallery or data/snapshot.png, or print omap.clusters.
MAP_FIXES = {
    "G21": (750.7, 1234.9),  # Queens Plaza (E M R): the circles its label's leader line points to
    "B04": (593.6, 1258.3),  # 21 St-Queensbridge (F)
    "F09": (654.5, 1180.5),  # Court Sq-23 St (E M)
    "H01": (1375.3, 900.0),  # Aqueduct Racetrack (A)
    "H03": (1402.1, 873.2),  # Howard Beach-JFK Airport (A)
    "Q01": (416.7, 816.9),   # Canal St (N Q): in the Canal St pill with the R W
}

---
## 2. The schedule (static GTFS)

The MTA publishes its schedule as a zip of CSV files in the GTFS format, which nearly every transit agency uses. This section downloads it, loads the tables we need and derives a few lookups that everything else relies on.

**Produces:** `routes`, `stops`, `trips`, `shapes`, `stop_times` (raw tables) · `stations` · `parent_of` · `station_name` · `stop_xy` · `COLOR` · `transfers` · `complex_of`

In [ ]:
def download_static(max_age_days=STATIC_MAX_AGE_DAYS):
    """Download the static GTFS zip into DATA, unless a copy younger than `max_age_days` is already there."""
    if STATIC_ZIP.exists() and time.time() - STATIC_ZIP.stat().st_mtime < max_age_days * 86400:
        print(f"Using cached {STATIC_ZIP}")
        return
    print("Downloading static GTFS…")
    r = requests.get(STATIC_URL, timeout=60)
    r.raise_for_status()  # stop with a clear error on 404/500 instead of saving an error page
    STATIC_ZIP.write_bytes(r.content)
    print(f"Saved {len(r.content) / 1e6:.1f} MB")


def read_table(name, **kwargs):
    """Read one GTFS table (e.g. "stops" → stops.txt) straight out of the zip.

    Every column is read as text on purpose: IDs like "101" or "A27" must never be turned into numbers,
    or joins between tables would silently fail. Numeric columns are converted explicitly afterwards.
    Extra keyword arguments (e.g. `usecols`) are passed to pandas.read_csv.
    """
    with zipfile.ZipFile(STATIC_ZIP) as z, z.open(f"{name}.txt") as f:
        return pd.read_csv(f, dtype=str, **kwargs)


def hms_to_seconds(col):
    """Convert a column of "HH:MM:SS" strings to seconds after midnight.

    GTFS times can exceed 24:00 for trips that run past midnight ("25:10:30" → 90630), so this
    deliberately does not use datetime parsing.
    """
    hms = col.str.split(":", expand=True).astype(int)
    return hms[0] * 3600 + hms[1] * 60 + hms[2]


download_static()

routes = read_table("routes")
stops = read_table("stops")
trips = read_table("trips")
shapes = read_table("shapes")
stop_times = read_table("stop_times", usecols=["trip_id", "stop_id", "arrival_time", "departure_time", "stop_sequence"])
transfers = read_table("transfers")

# Convert the columns that really are numbers.
stops[["stop_lat", "stop_lon"]] = stops[["stop_lat", "stop_lon"]].astype(float)
shapes[["shape_pt_lat", "shape_pt_lon"]] = shapes[["shape_pt_lat", "shape_pt_lon"]].astype(float)
shapes["shape_pt_sequence"] = shapes["shape_pt_sequence"].astype(int)
stop_times["stop_sequence"] = stop_times["stop_sequence"].astype(int)
stop_times["arr"] = hms_to_seconds(stop_times["arrival_time"])
stop_times["dep"] = hms_to_seconds(stop_times["departure_time"])

for name, df in [("routes", routes), ("stops", stops), ("trips", trips), ("shapes", shapes),
                 ("stop_times", stop_times), ("transfers", transfers)]:
    print(f"{name:11s} {len(df):>8,} rows")

### Lookups used everywhere

* **Coordinates in feet.** Latitude/longitude are angles, not distances (at NYC's latitude a degree of longitude is only ~0.76× a degree of latitude). We project to *NY State Plane, Long Island* (EPSG:2263), a flat grid measured in feet built for this area, so distances and lengths are real.
* **Stations vs platforms.** `stops.txt` lists every station three times: the parent station (`location_type == 1`) and one platform per direction (`N`/`S` suffix). `parent_of` maps any of them to the station.
* **Complexes.** Big hubs are split into one GTFS station per original line (Times Sq is 5 of them). `transfers.txt` links them; `complex_of` gives every station the ID of its complex, which the next-train feature will use.

In [ ]:
to_feet = Transformer.from_crs("EPSG:4326", "EPSG:2263", always_xy=True)  # lon/lat degrees → x/y feet
stops["x"], stops["y"] = to_feet.transform(stops["stop_lon"].values, stops["stop_lat"].values)
shapes["x"], shapes["y"] = to_feet.transform(shapes["shape_pt_lon"].values, shapes["shape_pt_lat"].values)

stations = stops[stops["location_type"] == "1"].copy()  # one row per station (parent stations only)

# Any stop_id → its station: "L27S" → "L27", and "L27" → "L27" (parents have no parent_station).
parent_of = {s: (p if isinstance(p, str) else s) for s, p in zip(stops["stop_id"], stops["parent_station"])}
station_name = dict(zip(stations["stop_id"], stations["stop_name"]))
stop_xy = dict(zip(stops["stop_id"], zip(stops["x"], stops["y"])))  # any stop_id → (x, y) in feet
COLOR = {r: "#" + c for r, c in zip(routes["route_id"], routes["route_color"])}  # route → official color

# Complexes: connected groups in the transfer graph. The complex ID is simply its smallest station ID.
transfer_graph = nx.Graph()
transfer_graph.add_nodes_from(stations["stop_id"])
transfer_graph.add_edges_from(zip(transfers["from_stop_id"], transfers["to_stop_id"]))
complex_of = {s: min(c) for c in nx.connected_components(transfer_graph) for s in c}

print(f"{len(stations)} stations in {len(set(complex_of.values()))} complexes, {len(routes)} routes")
stops[stops["stop_id"].str.startswith("L27")]  # one station = 1 parent + 2 platforms

---
## 3. Track geometry

The live feed never says *which* track a train is on, and its trip IDs (`127850_L..S`) don't match the schedule's (`ASP26GEN-1038-Sunday-00_000600_1..S03R`). What it does give is the **platform a train is heading to**, using the schedule's stop IDs. So we organize everything by pairs of consecutive platforms.

| Step | Result |
|---|---|
| 3a | **Patterns**: the distinct stop sequences trains run, with how many trips use each |
| 3b | **Segments**: the track between every pair of consecutive platforms, cut from the track shapes |
| 3c | **Hop times**: the scheduled travel time of each segment |
| 3d | Checks, simplification and a quick look |

**Produces:** `trip_stops` · `patterns` · `segments` (geometry, routes, trip count and travel time per platform pair)

In [ ]:
# 3a. Patterns. A shape is just a wiggly line; a pattern is the ordered platforms a train stops at.
trip_stops = (  # every scheduled stop of every trip, in order, with the trip's route and shape
    stop_times.merge(trips[["trip_id", "route_id", "shape_id"]], on="trip_id")
    .sort_values(["trip_id", "stop_sequence"])
)
trip_patterns = trip_stops.groupby("trip_id").agg(
    route_id=("route_id", "first"), shape_id=("shape_id", "first"), stops=("stop_id", tuple)
)
patterns = (  # one row per distinct (route, shape, stop sequence), busiest first
    trip_patterns.groupby(["route_id", "shape_id", "stops"]).size()
    .rename("n_trips").reset_index()
    .sort_values("n_trips", ascending=False, ignore_index=True)
)
shape_lines = {  # shape_id → LineString in feet
    sid: LineString(g.sort_values("shape_pt_sequence")[["x", "y"]].to_numpy())
    for sid, g in shapes.groupby("shape_id")
}
print(f"{len(trip_patterns):,} trips collapse into {len(patterns)} patterns over {len(shape_lines)} track shapes")
patterns.head(3)

In [ ]:
# 3b. Segments: cut each pattern's track shape at its stops.
def cut_at_stops(line, stop_ids):
    """Cut a track shape into the pieces between consecutive stops.

    Args:
        line: the pattern's track shape (LineString, feet).
        stop_ids: the pattern's platforms, in travel order.
    Yields:
        (from_stop, to_stop, piece) for each consecutive pair, `piece` being a LineString.

    For each stop we find the closest point on the line (`project` = distance along the line), but only
    searching *ahead of the previous stop*, so a line that doubles back near itself can't confuse us.
    """
    dists, start = [], 0.0
    for s in stop_ids:
        rest = substring(line, start, line.length)  # the part of the line still ahead
        d = start + (rest.project(Point(stop_xy[s])) if rest.length > 0 else 0.0)
        dists.append(d)
        start = d
    for a, b, d0, d1 in zip(stop_ids, stop_ids[1:], dists, dists[1:]):
        yield a, b, substring(line, d0, d1)


segments = {}  # (from_platform, to_platform) → {"geom", "routes", "n_trips", later "runtime" and "line"}
for p in patterns.itertuples():  # busiest patterns first, so they provide each segment's geometry
    for a, b, geom in cut_at_stops(shape_lines[p.shape_id], p.stops):
        seg = segments.setdefault((a, b), {"geom": geom, "routes": set(), "n_trips": 0})
        seg["routes"].add(p.route_id)
        seg["n_trips"] += p.n_trips
print(f"{len(segments):,} platform-to-platform segments")

In [ ]:
# 3c. Scheduled travel time of each segment: arrival at B minus departure from A, median over all trips.
# The live feed says when a train will *arrive* at its next stop, not when it left the last one;
# this time is what lets §6 work out how far along the segment it is.
next_stop = trip_stops.groupby("trip_id")[["stop_id", "arr"]].shift(-1)  # the following row within each trip
consecutive = pd.DataFrame({
    "a": trip_stops["stop_id"], "b": next_stop["stop_id"], "secs": next_stop["arr"] - trip_stops["dep"],
}).dropna()  # the last stop of each trip has no "next" row
runtime = consecutive.groupby(["a", "b"])["secs"].median()

for key, seg in segments.items():
    seg["runtime"] = float(runtime.get(key, np.nan))

runtimes_s = pd.Series({k: s["runtime"] for k, s in segments.items()})
print(f"Median segment time {runtimes_s.median():.0f} s; longest {runtimes_s.idxmax()} at {runtimes_s.max() / 60:.1f} min")

In [ ]:
# 3d. Checks: every stop should sit on its track, and no segment should have zero length.
snap_ft = pd.Series({b: Point(stop_xy[b]).distance(seg["geom"]) for (a, b), seg in segments.items()})
zero_len = [k for k, seg in segments.items() if seg["geom"].length < 1]
print(f"Stop-to-track distance: median {snap_ft.median():.0f} ft, worst {snap_ft.max():.0f} ft ({snap_ft.idxmax()})")
print(f"Zero-length segments: {len(zero_len)}")

# Simplify: Douglas–Peucker drops points closer than SIMPLIFY_TOL_FT to the line between their neighbors.
# Segment endpoints (the stations) are always kept.
for seg in segments.values():
    seg["line"] = seg["geom"].simplify(SIMPLIFY_TOL_FT)
before = sum(len(s["geom"].coords) for s in segments.values())
after = sum(len(s["line"].coords) for s in segments.values())
print(f"Simplified {before:,} → {after:,} points ({after / before:.0%})")

In [ ]:
def draw_geographic(ax, lw=1.0, station_size=2):
    """Draw the true-geography track map (feet) on `ax`, each segment in its route color.

    Where routes of different colors share a segment, the alphabetically first route's color is used.
    """
    ax.set_facecolor("black")
    for seg in segments.values():
        x, y = seg["line"].xy
        ax.plot(x, y, color=COLOR.get(min(seg["routes"]), "#888888"), lw=lw, solid_capstyle="round", zorder=1)
    ax.scatter(stations["x"], stations["y"], s=station_size, c="#555555", zorder=2, linewidths=0)
    ax.set_aspect("equal")
    ax.axis("off")


def bbox_around(lat, lon, half_width_ft):
    """Axis limits (xmin, xmax, ymin, ymax) in feet for a square centered on a latitude/longitude."""
    x, y = to_feet.transform(lon, lat)
    return x - half_width_ft, x + half_width_ft, y - half_width_ft, y + half_width_ft


fig, ax = plt.subplots(figsize=(7, 7), facecolor="black")
draw_geographic(ax)
ax.set_title("The track network in true geography (for reference)", color="white")
plt.show()

---
## 4. The official MTA map, on black

The MTA's subway diagram is a PDF of **vector artwork**: drawing instructions like *"move to (x, y), line to …, curve to …, stroke it orange, 8.2 pt wide"*. The `mtamap` package reads those instructions and redraws them with a black background, then works out where every GTFS station sits on the drawing.

> ⚠️ **Personal use only.** The map's small print says it *"may not be reproduced without written permission from the Metropolitan Transportation Authority and is not intended for resale."* Keep it on your own machine. If you ever share the screensaver, use Appendix A's generated map instead (or get MTA's permission).

### What `mtamap` does, and where to change it

| Step | What happens | Code |
|---|---|---|
| Read the PDF | Open it like any viewer (empty password), decompress the drawing streams | `mtamap/pdfreader.py` |
| Interpret | Turn drawing commands (`m` move, `l` line, `c` curve, `f` fill, `S` stroke, `Tj` text …) into shapes and labels | `mtamap/pdfdraw.py` |
| Recolor | Land, water, parks → black; black text → white; white outlines between lines → black; station circles stay black | `mtamap/style.py` |
| Route lines | Thick strokes in route colors, paired one-to-one with GTFS colors | `match.route_color_lines` |
| Station circles | Every route that stops gets a small black circle with its letter, on its own line | `match.station_circles` |
| Labels | Station names that are real text (two-line names joined) | `match.labels_index` |
| Match | A geography→map warp + the Hungarian algorithm assign stations to circle groups | `match.match_stations` |
| Read the letters | Learn which glyph shape is which route (votes weighted by trips/day), then match again | `match.learn_glyphs` |
| Fill + clean up | Walk along routes to fill gaps; swap/move stations when that makes routes shorter | `match.complete_chains`, `match.refine` |

**Produces:** `omap`, an `mtamap.OfficialMap` with:
* `omap.station_point(route, station)`: the map position of that route's circle at that station
* `omap.hop_line(route, a, b)`: the stretch of the route's colored line from station a to b
* `omap.draw(ax)`: draws the map

In [ ]:
def download_map():
    """Download the official diagram PDF into DATA unless it's already there.

    Unlike the schedule this isn't refreshed automatically: delete data/subway_diagram.pdf to fetch a new
    edition (then check §4's matching report, since a redesigned map may need adjustments).
    """
    if not MAP_PDF.exists():
        r = requests.get(MAP_URL, timeout=60)
        r.raise_for_status()
        MAP_PDF.write_bytes(r.content)
    print(f"{MAP_PDF}: {MAP_PDF.stat().st_size / 1e3:.0f} KB")


download_map()
omap = mtamap.OfficialMap(MAP_PDF, stops, stations, patterns, COLOR, complex_of=complex_of,
                          fixes=MAP_FIXES)  # prints a matching report

In [ ]:
fig = plt.figure(figsize=(12, 12 * mtamap.MAP_H / mtamap.PAGE_W), facecolor="black")
omap.draw(fig.add_axes([0, 0, 1, 1]))
plt.show()

### Checking the matching by eye

`check_matching` zooms into a box of the map and writes each GTFS station's name (cyan) where it was matched. A red ring marks a circle group no station claimed, which is usually a sign of a mismatch nearby. Map coordinates run x 0–1656 (left→right) and y 0–1978 (bottom→top).

In [ ]:
def check_matching(x0, x1, y0, y1, scale=3):
    """Draw the map inside x0..x1, y0..y1 with matched GTFS station names overlaid.

    Args:
        x0, x1, y0, y1: the box to show, in map coordinates.
        scale: figure size multiplier (bigger = more readable, slower).
    """
    owners = defaultdict(list)  # circle group → stations matched to it
    for sid, k in omap.assigned.items():
        owners[k].append(sid)
    fig = plt.figure(figsize=((x1 - x0) / 100 * scale, (y1 - y0) / 100 * scale), facecolor="black")
    ax = fig.add_axes([0, 0, 1, 1])
    omap.draw(ax, xlim=(x0, x1), ylim=(y0, y1))
    for k, group in enumerate(omap.clusters):
        x, y = group["center"]
        if not (x0 < x < x1 and y0 < y < y1):
            continue
        if k in owners:
            label = ", ".join(omap.names[s][:16] for s in owners[k])
            ax.text(x + 3, y - 9, label, color="cyan", fontsize=4.5 * scale, weight="bold", zorder=9)
        else:
            ax.plot(x, y, "o", mfc="none", mec="red", ms=4 * scale, mew=2, zorder=9)
    plt.show()


check_matching(120, 480, 900, 1250)  # Midtown Manhattan

---
## 5. Live feeds (GTFS-Realtime)

The live data comes as 8 feeds, one per group of lines. Each is a **Protocol Buffer** (a compact binary format); `gtfs-realtime-bindings` decodes it into Python objects. Two kinds of record matter to us, linked by `trip_id`:

| Record | Tells us | Example |
|---|---|---|
| `trip_update` | Predicted times at the stops **still ahead** (stops already passed are dropped) | L27S at 21:54:02, L28S at 21:55:59 … |
| `vehicle` | Where the train is now, as a platform + status. **No GPS.** | `IN_TRANSIT_TO L27S` |

`fetch_all()` returns one row per trip:

| Column | Meaning |
|---|---|
| `feed` | Which of the 8 feeds it came from |
| `trip_id` | Live trip ID (does **not** match the schedule's trip IDs) |
| `route_id` | Route, e.g. `L` |
| `direction` | `N` or `S`, from the first upcoming platform |
| `status` | `STOPPED_AT`, `INCOMING_AT`, `IN_TRANSIT_TO`, or `None` if no train is assigned yet |
| `vehicle_stop` | The platform the status refers to, or `None` |
| `upcoming` | List of `(platform, unix_time)` for the stops still ahead |

**Produces:** `trains` (DataFrame) · `fetched_at` (unix time of the download)

In [ ]:
STATUS = {0: "INCOMING_AT", 1: "STOPPED_AT", 2: "IN_TRANSIT_TO"}  # GTFS-Realtime vehicle status codes


def fetch_feed(name):
    """Download one realtime feed (e.g. "gtfs-l") and return it decoded as a FeedMessage."""
    r = requests.get(FEED_URL.format(name), timeout=10)
    r.raise_for_status()
    msg = gtfs_realtime_pb2.FeedMessage()
    msg.ParseFromString(r.content)
    return msg


def fetch_all():
    """Download all realtime feeds and combine them into one table.

    Returns:
        (trains, fetched_at): a DataFrame with one row per trip (columns described above) and the unix
        time just before downloading, which §6 uses as "now".
    """
    fetched_at = time.time()
    rows = []
    for name in FEEDS:
        msg = fetch_feed(name)
        updates = {e.trip_update.trip.trip_id: e.trip_update for e in msg.entity if e.HasField("trip_update")}
        vehicles = {e.vehicle.trip.trip_id: e.vehicle for e in msg.entity if e.HasField("vehicle")}
        for trip_id, tu in updates.items():  # every trip has a trip_update; only some have a vehicle
            v = vehicles.get(trip_id)
            upcoming = [
                # Use the arrival time, or the departure time at a trip's first stop (which has no arrival).
                (s.stop_id, s.arrival.time if s.HasField("arrival") else s.departure.time)
                for s in tu.stop_time_update
            ]
            rows.append({
                "feed": name,
                "trip_id": trip_id,
                "route_id": tu.trip.route_id,
                "direction": upcoming[0][0][-1] if upcoming else None,  # last letter of the platform ID
                "status": STATUS[v.current_status] if v else None,
                "vehicle_stop": v.stop_id if v else None,
                "upcoming": upcoming,
            })
    return pd.DataFrame(rows), fetched_at


trains, fetched_at = fetch_all()
print(f"{len(trains)} trips at {time.strftime('%-I:%M:%S %p', time.localtime(fetched_at))}")
print(trains["status"].value_counts(dropna=False).to_string())
trains.head()

---
## 6. Placing every train on the map

`locate()` decides, for each trip, *whether* to draw it and *where*. It works through these checks in order; the first that applies decides:

| # | Situation | Result (`how`) | Drawn? |
|---|---|---|---|
| 1 | No stops left and the train is stopped | `parked (finished run)` at its last station | dimmed |
| 1 | No stops left otherwise | `no predictions` | no |
| 2 | Next predicted time is more than `STALE_AFTER_S` ago | `stale` (a leftover trip) | no |
| 3 | The platform isn't in the schedule or on the map | `unknown stop` (e.g. phantom `H19`) | no |
| 4 | `STOPPED_AT` a platform | `at station`, on that route's circle | yes |
| 5 | No train assigned yet, or heading to the first stop of its run | `parked (leaving soon)` if it departs within `PARK_WINDOW_S`, else `waiting (hidden)` | dimmed / no |
| 6 | Otherwise it's moving between two stops | `between stops` along the route's line, or `between stops (straight)` if the line couldn't be followed | yes |

**Where along the hop?** The feed says when the train will reach its next stop. Compare that with the hop's normal travel time:

$$\text{fraction traveled} = 1 - \frac{\text{seconds until arrival}}{\text{scheduled hop time}} \quad\text{(clipped to 0–1)}$$

Due in 30 s on a 120 s hop means ¾ of the way there.

**Which stop did it just leave?** The feed never says. `previous_stop()` compares the train's upcoming stops with each of the route's patterns and picks the best match. Example: a northbound 2 reaching 14 St comes from Chambers St (express) by day, but from Christopher St (local) between midnight and ~7 AM. If its next stops are 18 St, 23 St … it's running local. Known limit: a northbound A at Rockaway Blvd has identical upcoming stops from either branch; that needs memory of earlier positions (§7).

**Produces:** new columns on `trains`: `x`, `y` (map coordinates, NaN when not drawn) and `how` (the category above) · `data/snapshot.png`

In [ ]:
route_patterns = defaultdict(list)  # route → [(platform sequence, trips per day), ...]
prev_any = defaultdict(Counter)  # platform → Counter of the platforms that come before it, on any route
for p in patterns.itertuples():
    route_patterns[p.route_id].append((p.stops, p.n_trips))
    for a, b in zip(p.stops, p.stops[1:]):
        prev_any[b][a] += p.n_trips


def previous_stop(route, stop, upcoming_stops):
    """Guess the platform a train left before heading to `stop`.

    Args:
        route: the train's route ID.
        stop: the platform it is heading to.
        upcoming_stops: its upcoming platforms in order (from the feed).
    Returns:
        The previous platform ID, or None if `stop` is the first stop of the best-matching pattern
        (the train hasn't started its run).

    Each of the route's patterns containing `stop` is scored by how many of the train's next 8 stops
    it also visits after `stop`; ties go to the pattern with more trips per day.
    """
    best, best_key = None, None
    for stops_, n_trips in route_patterns.get(route, []):
        if stop not in stops_:
            continue
        i = stops_.index(stop)
        ahead = set(stops_[i:])
        key = (sum(s in ahead for s in upcoming_stops[:8]), n_trips)  # best match first, then most common
        if best_key is None or key > best_key:
            best_key, best = key, (stops_[i - 1] if i > 0 else None)
    if best_key is None:
        # This route never stops here in the schedule (a reroute): take the most common predecessor on any route.
        candidates = prev_any.get(stop)
        return candidates.most_common(1)[0][0] if candidates else None
    return best


def station_point(route, stop_id):
    """Map position of `route`'s circle at the station of platform `stop_id`."""
    return omap.station_point(route, parent_of[stop_id])


def hop_fraction(prev, stop, eta, now):
    """How far (0–1) a train has traveled from platform `prev` to `stop`, given it arrives at unix time `eta`."""
    seg = segments.get((prev, stop))
    hop_time = seg["runtime"] if seg and seg["runtime"] > 0 else DEFAULT_HOP_S
    return min(max(1 - (eta - now) / hop_time, 0.0), 1.0)


def locate(train, now):
    """Decide whether and where to draw one trip (see the table above).

    Args:
        train: one row of `trains`.
        now: unix time to evaluate predictions against (normally `fetched_at`).
    Returns:
        (x, y, how): map coordinates (NaN if not drawn) and the category label.
    """
    nan = (np.nan, np.nan)

    # 1. No stops left: the run is over.
    if not train["upcoming"]:
        stop = train["vehicle_stop"]
        if train["status"] == "STOPPED_AT" and parent_of.get(stop) in omap.assigned:
            return (*station_point(train["route_id"], stop), "parked (finished run)")
        return (*nan, "no predictions")

    # 2. Leftover trips whose "next" stop time is long past.
    first_stop, first_time = train["upcoming"][0]
    if first_time < now - STALE_AFTER_S:
        return (*nan, "stale")

    # 3. A platform we can't place (not in the schedule, or not found on the map).
    route = train["route_id"]
    stop = train["vehicle_stop"] or first_stop  # trips without a vehicle: use their first upcoming stop
    if parent_of.get(stop) not in omap.assigned:
        return (*nan, "unknown stop")

    # 4. Standing at a platform.
    if train["status"] == "STOPPED_AT":
        return (*station_point(route, stop), "at station")

    # 5. Not started yet: no train assigned, or heading to the first stop of its run.
    prev = previous_stop(route, stop, [s for s, _ in train["upcoming"]])
    if train["status"] is None or prev is None:
        if first_time - now <= PARK_WINDOW_S:
            return (*station_point(route, stop), "parked (leaving soon)")
        return (*nan, "waiting (hidden)")

    # 6. Moving: place it along the route's line between the two stations.
    line, followed = omap.hop_line(route, parent_of[prev], parent_of[stop])
    if line is None:
        return (*station_point(route, stop), "at next stop (no hop)")
    eta = dict(train["upcoming"]).get(stop, first_time)
    pt = line.interpolate(hop_fraction(prev, stop, eta, now), normalized=True)
    return (pt.x, pt.y, "between stops" if followed else "between stops (straight)")


located = pd.DataFrame(
    [locate(t, fetched_at) for _, t in trains.iterrows()], columns=["x", "y", "how"], index=trains.index
)
trains[["x", "y", "how"]] = located
print(trains["how"].value_counts().to_string())

In [ ]:
def draw_trains(ax, df, size=18):
    """Draw located trains on a map axis.

    * Moving trains: solid white dots with a thin black edge, which stand out on a line of any color.
    * Trains standing at a station: a white ring *around* that route's circle, so its letter stays readable.
    * Parked trains (finished, or about to leave their terminal): a dim gray ring.

    Args:
        ax: axis already showing the map (from `omap.draw`).
        df: `trains` after §6, needing columns x, y, how.
        size: marker area in points² of a moving-train dot (scale up for zoomed views; rings scale with it).
    """
    df = df.dropna(subset=["x", "y"])
    parked = df["how"].str.startswith("parked")
    at_station = df["how"].isin(["at station", "at next stop (no hop)"])
    moving = ~parked & ~at_station
    edge = size ** 0.5 / 8  # outline width in points, proportional to the dot's diameter
    ring = dict(s=size * 2.6, facecolors="none", linewidths=edge * 2.5)  # ring diameter ≈ 1.6 × dot diameter
    ax.scatter(df.loc[parked, "x"], df.loc[parked, "y"], edgecolors="#777777", zorder=10, **ring)
    ax.scatter(df.loc[at_station, "x"], df.loc[at_station, "y"], edgecolors="white", zorder=11, **ring)
    ax.scatter(df.loc[moving, "x"], df.loc[moving, "y"], s=size, c="white", edgecolors="black",
               linewidths=edge, zorder=12)


ZOOM = dict(xlim=(100, 560), ylim=(560, 1110))  # Lower & Midtown Manhattan + Downtown Brooklyn

stamp = time.strftime("%-I:%M:%S %p", time.localtime(fetched_at))
panel_w = 11  # inches per panel
fig = plt.figure(figsize=(2 * panel_w, panel_w * mtamap.MAP_H / mtamap.PAGE_W), facecolor="black")
ax_all, ax_zoom = fig.add_axes([0, 0, 0.5, 1]), fig.add_axes([0.5, 0, 0.5, 1])
omap.draw(ax_all)
draw_trains(ax_all, trains, size=22)
ax_all.set_title(f"Every train, {stamp}", color="white", y=0.97)
omap.draw(ax_zoom, **ZOOM)
draw_trains(ax_zoom, trains, size=90)
fig.savefig(DATA / "snapshot.png", dpi=110, facecolor="black")
plt.show()

---
## 7. Focused views

A focused view crops the map to a handful of stations, framed to the exact shape of your screen, with live trains and the chosen stations outlined. These are candidates for a "zoomed" screensaver mode (e.g. your neighborhood, or rotating through a few views).

| Function | What it does |
|---|---|
| `find_stations("text")` | Lists stations whose name contains the text, with IDs and routes, to help you fill in `FOCUS_VIEWS` |
| `resolve_stations([...])` | Turns names/IDs into station IDs (adding each station's complex), and stops with a clear message if a name is ambiguous |
| `focus_box(ids)` | The map rectangle that holds those stations plus `FOCUS_PAD`, stretched to `SCREEN_ASPECT` |
| `render_focus(ax, ids, title)` | Draws a focused view into an existing axis |
| `draw_focus(view)` | One view as its own screen-shaped figure, saved to `data/focus_<name>.png` |

Presets live in `FOCUS_VIEWS` in the settings cell (§1). To add one, add a line there and re-run §1 and §7.

**Produces:** the functions above · `data/focus_*.png`

In [ ]:
from matplotlib.patches import FancyBboxPatch

station_routes = omap.served  # station → set of routes that stop there (from the schedule)
complex_members = defaultdict(list)  # complex ID → its stations
for s, c in complex_of.items():
    complex_members[c].append(s)


def find_stations(text):
    """Stations whose name contains `text` (case-insensitive), as a table of ID, name, routes and complex.

    Handy for filling in FOCUS_VIEWS: use the name if it's unique, otherwise the ID.
    """
    rows = [
        {"id": s, "name": n, "routes": " ".join(sorted(station_routes.get(s, []))), "complex": complex_of.get(s)}
        for s, n in station_name.items() if text.lower() in n.lower() and s in omap.assigned
    ]
    return pd.DataFrame(rows).sort_values(["name", "id"], ignore_index=True)


def resolve_stations(items):
    """Turn station IDs and/or exact station names into a list of station IDs on the map.

    Args:
        items: e.g. ["Times Sq-42 St", "A38"]. Names are matched case-insensitively and must be exact.
    Returns:
        Station IDs in order, without duplicates, each followed by the other stations in its complex.
    Raises:
        ValueError: if a name matches nothing, or matches stations in different places
            (more than SAME_NAME_RADIUS from their common middle on the map). The message lists the IDs to use.
    """
    ids = []
    for item in items:
        if item in station_name:  # already a station ID
            matches = [item]
        else:
            matches = [s for s, n in station_name.items() if n.lower() == item.lower() and s in omap.assigned]
            if not matches:
                raise ValueError(f"No station named {item!r}. Try find_stations({item.split()[0]!r}) to see names.")
            centers = np.array([omap.clusters[omap.assigned[s]]["center"] for s in matches])
            spread = np.linalg.norm(centers - centers.mean(axis=0), axis=1).max()  # farthest from their middle
            if spread > SAME_NAME_RADIUS:
                options = ", ".join(f"{s} ({' '.join(sorted(station_routes.get(s, [])))})" for s in matches)
                raise ValueError(f"{item!r} names stations in different places; use an ID instead: {options}")
        for s in matches:
            ids.append(s)
            ids += sorted(m for m in complex_members[complex_of.get(s, s)] if m in omap.assigned)  # whole complex
    return list(dict.fromkeys(ids))  # drop duplicates, keep order


def focus_box(station_ids, pad=FOCUS_PAD, aspect=SCREEN_ASPECT, min_width=FOCUS_MIN_WIDTH):
    """The map rectangle (x0, x1, y0, y1) for a focused view.

    Starts from the bounding box of every circle drawn for the stations, adds `pad` on all sides, makes it at
    least `min_width` wide, then grows the short side (keeping the center) until width ÷ height == `aspect`.
    """
    boxes = np.array([omap.station_extent(s) for s in station_ids])  # rows: xmin, ymin, xmax, ymax
    x0, y0 = boxes[:, 0].min() - pad, boxes[:, 1].min() - pad
    x1, y1 = boxes[:, 2].max() + pad, boxes[:, 3].max() + pad
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    w, h = max(x1 - x0, min_width), y1 - y0
    if w / h < aspect:  # too tall for the screen: widen
        w = h * aspect
    else:  # too wide: heighten
        h = w / aspect
    return cx - w / 2, cx + w / 2, cy - h / 2, cy + h / 2


def render_focus(ax, station_ids, title):
    """Draw a focused view into `ax`.

    Draws the cropped map, a dashed outline around each chosen station, the live trains (sized in map units so
    they look the same at any zoom) and a title in the top-left corner.
    """
    x0, x1, y0, y1 = focus_box(station_ids)
    pts_per_unit = omap.draw(ax, xlim=(x0, x1), ylim=(y0, y1))  # how many points one map unit becomes
    for s in station_ids:
        bx0, by0, bx1, by1 = omap.station_extent(s)
        ax.add_patch(FancyBboxPatch((bx0, by0), bx1 - bx0, by1 - by0, boxstyle="round,pad=3,rounding_size=7",
                                    fill=False, ec="white", lw=0.35 * pts_per_unit, ls=(0, (3, 2)), zorder=9))
    draw_trains(ax, trains, size=(TRAIN_DOT_UNITS * pts_per_unit) ** 2)  # marker area is in points²
    ax.text(0.015, 0.975, title, transform=ax.transAxes, ha="left", va="top", color="white",
            fontsize=max(9, 7 * pts_per_unit), weight="bold", family=["Helvetica", "Arial", "DejaVu Sans"],
            bbox=dict(facecolor="black", edgecolor="none", pad=4), zorder=12)


def draw_focus(view, width_in=12, save=True):
    """Draw one focused view as a screen-shaped figure.

    Args:
        view: a FOCUS_VIEWS name (e.g. "Midtown"), or a list of station names/IDs for a one-off view.
        width_in: figure width in inches (the height follows SCREEN_ASPECT).
        save: also write data/focus_<name>.png.
    Returns:
        The matplotlib Figure.
    """
    name, items = (view, FOCUS_VIEWS[view]) if isinstance(view, str) else ("Custom view", view)
    ids = resolve_stations(items)
    fig = plt.figure(figsize=(width_in, width_in / SCREEN_ASPECT), facecolor="black")
    stamp = time.strftime("%-I:%M %p", time.localtime(fetched_at))
    render_focus(fig.add_axes([0, 0, 1, 1]), ids, f"{name}  ·  {stamp}")
    if save:
        fig.savefig(DATA / f"focus_{name.lower().replace(' ', '_')}.png", dpi=150, facecolor="black")
    return fig


draw_focus("Midtown")
plt.show()

### All presets

Each tile is framed exactly like your screen. Edit `FOCUS_VIEWS` in §1 and re-run to change them.

In [ ]:
view_names = list(FOCUS_VIEWS)
cols = 2
n_rows = -(-len(view_names) // cols)  # ceiling division
tile_w = 8  # inches per tile
fig = plt.figure(figsize=(cols * tile_w, n_rows * tile_w / SCREEN_ASPECT), facecolor="black")
for i, name in enumerate(view_names):
    r, c = divmod(i, cols)
    gap = 0.004  # thin black gutter between tiles (fraction of the figure)
    ax = fig.add_axes([c / cols + gap, 1 - (r + 1) / n_rows + gap, 1 / cols - 2 * gap, 1 / n_rows - 2 * gap])
    render_focus(ax, resolve_stations(FOCUS_VIEWS[name]), name)
fig.savefig(DATA / "focus_gallery.png", dpi=110, facecolor="black")
plt.show()

### Finding stations, and what an ambiguous name looks like

In [ ]:
print(find_stations("fulton").to_string(index=False))
print()
try:
    resolve_stations(["Fulton St"])  # five stations share this name; one is in Brooklyn
except ValueError as err:
    print("ValueError:", err)

---
## 8. Export for the Swift screensaver

The screensaver shouldn't redo sections 2–4 every time it starts (that's most of this notebook's 30 seconds). Instead this section saves their results, and Swift only loads files, draws, downloads the live feeds and animates.

| File | Contents |
|---|---|
| `export/map_black.pdf` | The black map as **vector** artwork (crisp at any zoom, for focused views) |
| `export/network.json` | Everything `locate()` needs: stations and their per-route circle positions, platform → station, stop patterns, scheduled hop times, the colored route lines, focus views and the §1 settings. Plus each station's official **direction labels** (from the MTA's station list) for the next-train panel |

**Coordinates.** Both files share one coordinate system: the map's, with x 0–1656 left→right and y 0–1978 bottom→top. The PDF page is exactly that size in points, so a position in `network.json` can be drawn straight onto the PDF.

**When to re-run:** when the MTA updates the schedule (a few times a year; see `feed_end_date` below) or redesigns the map. Run the whole notebook; this section rewrites both files.

> ⚠️ `map_black.pdf` is MTA artwork recolored: personal use only, like §4. Don't share it.

**Produces:** `export/map_black.pdf` · `export/network.json` · `network` (the same data as a Python dict)

In [ ]:
import json

EXPORT_DIR.mkdir(exist_ok=True)
EXPORT_PDF = EXPORT_DIR / "map_black.pdf"
EXPORT_JSON = EXPORT_DIR / "network.json"


def export_map_pdf(path):
    """Write the black map as a vector PDF whose page coordinates are the map coordinates.

    The page is PAGE_W × MAP_H points, one map unit = one PDF point, origin at the bottom-left, so
    `omap.draw` reports exactly 1 point per map unit and nothing is scaled.
    """
    fig = plt.figure(figsize=(mtamap.PAGE_W / 72, mtamap.MAP_H / 72), facecolor="black")  # 72 points per inch
    pts_per_unit = omap.draw(fig.add_axes([0, 0, 1, 1]))
    assert abs(pts_per_unit - 1) < 1e-9, pts_per_unit  # page and map coordinates must agree
    fig.savefig(path, facecolor="black")
    plt.close(fig)


def r1(v):
    """Round a coordinate to 0.1 map units (far below one screen pixel) to keep the JSON small."""
    return round(float(v), 1)


def download_station_list(max_age_days=STATIC_MAX_AGE_DAYS):
    """Download the MTA's station list (direction labels, borough…) into DATA unless a recent copy exists."""
    if STATIONS_CSV.exists() and time.time() - STATIONS_CSV.stat().st_mtime < max_age_days * 86400:
        return
    r = requests.get(STATIONS_URL, timeout=60)
    r.raise_for_status()
    STATIONS_CSV.write_bytes(r.content)


download_station_list()
station_list = pd.read_csv(STATIONS_CSV, dtype=str).set_index("GTFS Stop ID")  # one row per GTFS station


def station_record(s):
    """Everything the screensaver needs to know about one station.

    `points` holds where each route's train should be drawn at this station: exactly what
    `omap.station_point(route, s)` returns, for every route that could show up here (routes that stop here,
    routes whose letter appears in its circles, and routes whose line color passes under them).
    Any other route falls back to `center`, as `omap.station_point` does.
    """
    members = omap.clusters[omap.assigned[s]]["members"]
    candidates = set(station_routes.get(s, ()))
    for i in members:
        candidates |= omap.circle_routes(i)  # letters drawn in this station's circles
        candidates |= {r for r, col in COLOR.items() if col in omap.circles[i]["lines"]}  # lines passing under
    cx, cy = omap.clusters[omap.assigned[s]]["center"]
    return {
        "name": station_name[s],
        "complex": complex_of.get(s, s),
        "routes": sorted(station_routes.get(s, ())),
        "center": [r1(cx), r1(cy)],
        "extent": [r1(v) for v in omap.station_extent(s)],  # xmin, ymin, xmax, ymax of its circles
        "points": {r: [r1(v) for v in omap.station_point(r, s)] for r in sorted(candidates)},
        # Official labels for trains toward the N and S platforms (e.g. "Uptown" / "Downtown",
        # "Manhattan" / "Coney Island"). Fall back to plain compass words if the MTA list lacks the station.
        "north_label": station_list["North Direction Label"].get(s, "Northbound"),
        "south_label": station_list["South Direction Label"].get(s, "Southbound"),
        "borough": station_list["Borough"].get(s),
    }



def hop_paths_for_export(stations):
    """The track a train follows for every hop it can make, computed once here (omap.hop_line) so Swift draws
    exactly the same paths: "route|from_station|to_station" → points. Covers each route's own hops, plus hops
    between consecutive stations of any pattern for every route drawn at both (rerouted trains). Hops that
    can't follow the track are left out; the reader then uses a straight line.
    """
    pairs = set()
    for p in patterns.itertuples():
        seq = [parent_of[s] for s in p.stops]
        pairs |= {(p.route_id, a, b) for a, b in zip(seq, seq[1:]) if a != b}
    for a, b in {(a, b) for _, a, b in pairs}:
        if a in stations and b in stations:
            drawn = set(stations[a]["points"]) & set(stations[b]["points"])
            serving = set(stations[a]["routes"]) | set(stations[b]["routes"])
            pairs |= {(r, a, b) for r in drawn | serving}
    paths = {}
    for r, a, b in sorted(pairs):
        line, followed = omap.hop_line(r, a, b)
        if followed:
            paths[f"{r}|{a}|{b}"] = [[r1(x), r1(y)] for x, y in line.coords]
    return paths

feed_info = read_table("feed_info").iloc[0].to_dict()  # the schedule's version and validity dates

network = {
    "meta": {
        "generated_at": time.strftime("%Y-%m-%dT%H:%M:%S%z"),
        "schedule_version": feed_info.get("feed_version"),
        "feed_start_date": feed_info.get("feed_start_date"),  # YYYYMMDD
        "feed_end_date": feed_info.get("feed_end_date"),  # after this date, re-run the notebook
        "map_pdf": EXPORT_PDF.name,
        "coordinates": "map units = PDF points; x 0–1656 left→right, y 0–1978 bottom→top",
        "map_size": [mtamap.PAGE_W, mtamap.MAP_H],
    },
    "settings": {  # from §1, so Swift behaves exactly like this notebook
        "stale_after_s": STALE_AFTER_S, "park_window_s": PARK_WINDOW_S, "default_hop_s": DEFAULT_HOP_S,
        "train_dot_units": TRAIN_DOT_UNITS, "focus_pad": FOCUS_PAD, "focus_min_width": FOCUS_MIN_WIDTH,
        "feed_url": FEED_URL, "feeds": FEEDS,
    },
    "routes": {  # route → colors and long name
        r.route_id: {"color": "#" + r.route_color, "text_color": "#" + r.route_text_color, "name": r.route_long_name}
        for r in routes.itertuples()
    },
    "platforms": {s: parent_of[s] for s in stops["stop_id"]},  # any stop ID (platform or station) → station
    "stations": {s: station_record(s) for s in sorted(omap.assigned)},
    "patterns": [  # for previous_stop(): every route's stop sequences, with trips per day
        {"route": p.route_id, "stops": list(p.stops), "trips": int(p.n_trips)} for p in patterns.itertuples()
    ],
    "hop_times": {  # "fromPlatform>toPlatform" → scheduled seconds (missing = use default_hop_s)
        f"{a}>{b}": seg["runtime"] for (a, b), seg in segments.items() if seg["runtime"] > 0
    },
    "route_lines": {  # line color → its drawn polylines, which trains follow between stations
        color: [[[r1(x), r1(y)] for x, y in line] for line in lines] for color, lines in omap.lines.items()
    },
    "focus_views": {name: resolve_stations(items) for name, items in FOCUS_VIEWS.items()},
}
network["hop_paths"] = hop_paths_for_export(network["stations"])  # "route|from|to" → the track followed
network["route_strokes"] = [  # each route line as the map draws it, in drawing order: later strokes are on top
    {"order": order, "color": color, "half_width": round(width / 2, 2), "points": [[r1(x), r1(y)] for x, y in line]}
    for order, color, width, polylines in mtamap.match.route_strokes(omap.drawing, COLOR)[0] for line in polylines
]
network["pair_paths"] = {  # "from|to" → the track between neighboring stations, for trains rerouted onto it
    f"{a}|{b}": [[r1(x), r1(y)] for x, y in line.coords]
    for (a, b) in sorted(omap.routes_between) if (line := omap.pair_line(a, b)) is not None
}

export_map_pdf(EXPORT_PDF)
EXPORT_JSON.write_text(json.dumps(network, separators=(",", ":")))  # compact: no spaces or line breaks

for path in (EXPORT_PDF, EXPORT_JSON):
    print(f"{path}: {path.stat().st_size / 1e3:,.0f} KB")
print(f"{len(network['stations'])} stations, {len(network['platforms']):,} platforms, {len(network['patterns'])} patterns, "
      f"{len(network['hop_times']):,} hop times, {sum(map(len, network['route_lines'].values()))} route-line pieces, "
      f"{len(network['hop_paths']):,} hop paths, {len(network['pair_paths']):,} station-pair paths, "
      f"{len(network['route_strokes'])} route strokes")
print(f"Schedule valid {feed_info['feed_start_date']} – {feed_info['feed_end_date']} (version {feed_info['feed_version']})")
missing_labels = [s for s in network["stations"] if s not in station_list.index]
print(f"Direction labels: {len(network['stations']) - len(missing_labels)} of {len(network['stations'])} stations "
      f"from the MTA list" + (f" (missing: {missing_labels})" if missing_labels else ""))

### Check: can the export alone place every train?

This is a dress rehearsal for the Swift port. `locate_from_export` repeats §6's `locate()` step by step, but reads **only** `network.json` (loaded fresh from disk) instead of the notebook's tables and `omap`. If the export is complete, every train gets the same category and the same position (within the 0.1-unit rounding) as in §6.

In [ ]:
net = json.loads(EXPORT_JSON.read_text())  # read back from disk, exactly what Swift will load

# Lookups built from the export alone (Swift will build the same ones when it starts).
net_patterns = defaultdict(list)  # route → [(stops, trips)]
net_prev_any = defaultdict(Counter)  # platform → Counter of preceding platforms, any route
for p in net["patterns"]:
    net_patterns[p["route"]].append((p["stops"], p["trips"]))
    for a, b in zip(p["stops"], p["stops"][1:]):
        net_prev_any[b][a] += p["trips"]
net_lines = {color: [np.array(line) for line in lines] for color, lines in net["route_lines"].items()}
net_cfg = net["settings"]


def net_previous_stop(route, stop, upcoming_stops):
    """Same as §6 previous_stop(), using the exported patterns."""
    best, best_key = None, None
    for stops_, n_trips in net_patterns.get(route, []):
        if stop not in stops_:
            continue
        i = stops_.index(stop)
        ahead = set(stops_[i:])
        key = (sum(s in ahead for s in upcoming_stops[:8]), n_trips)
        if best_key is None or key > best_key:
            best_key, best = key, (stops_[i - 1] if i > 0 else None)
    if best_key is None:
        candidates = net_prev_any.get(stop)
        return candidates.most_common(1)[0][0] if candidates else None
    return best


def net_station_point(route, station):
    """Same as omap.station_point(): the route's circle at the station, else the station's center."""
    rec = net["stations"][station]
    return tuple(rec["points"].get(route, rec["center"]))


def locate_from_export(train, now):
    """§6 locate(), steps 1–6, reading only `net`. Returns (x, y, how)."""
    nan = (np.nan, np.nan)
    plat = net["platforms"]
    if not train["upcoming"]:  # 1
        stop = train["vehicle_stop"]
        if train["status"] == "STOPPED_AT" and plat.get(stop) in net["stations"]:
            return (*net_station_point(train["route_id"], plat[stop]), "parked (finished run)")
        return (*nan, "no predictions")
    first_stop, first_time = train["upcoming"][0]
    if first_time < now - net_cfg["stale_after_s"]:  # 2
        return (*nan, "stale")
    route = train["route_id"]
    stop = train["vehicle_stop"] or first_stop
    if plat.get(stop) not in net["stations"]:  # 3
        return (*nan, "unknown stop")
    if train["status"] == "STOPPED_AT":  # 4
        return (*net_station_point(route, plat[stop]), "at station")
    prev = net_previous_stop(route, stop, [s for s, _ in train["upcoming"]])
    if train["status"] is None or prev is None:  # 5
        if first_time - now <= net_cfg["park_window_s"]:
            return (*net_station_point(route, plat[stop]), "parked (leaving soon)")
        return (*nan, "waiting (hidden)")
    pa, pb = net_station_point(route, plat[prev]), net_station_point(route, plat[stop])  # 6
    # The exported track, as Swift reads it: this route's own path, else (a rerouted train) the track of a route
    # that normally runs between the two stations, else its own colored line computed here, else a straight line.
    path = net["hop_paths"].get(f"{route}|{plat[prev]}|{plat[stop]}")
    if path:
        line, followed = LineString(path), True
    else:
        line, followed = mtamap.match.hop_line(pa, pb, net_lines.get(net["routes"].get(route, {}).get("color"), []))
        pair = net["pair_paths"].get(f"{plat[prev]}|{plat[stop]}")
        if not followed and pair:
            line, followed = LineString(pair), True
    hop_time = net["hop_times"].get(f"{prev}>{stop}", net_cfg["default_hop_s"])
    eta = dict(train["upcoming"]).get(stop, first_time)
    frac = min(max(1 - (eta - now) / hop_time, 0.0), 1.0)
    pt = line.interpolate(frac, normalized=True)
    return (pt.x, pt.y, "between stops" if followed else "between stops (straight)")


check = pd.DataFrame([locate_from_export(t, fetched_at) for _, t in trains.iterrows()],
                     columns=["x", "y", "how"], index=trains.index)
same_how = (check["how"] == trains["how"]).mean()
drawn = trains["x"].notna() & check["x"].notna()
dist = np.hypot(check.loc[drawn, "x"] - trains.loc[drawn, "x"], check.loc[drawn, "y"] - trains.loc[drawn, "y"])
print(f"Same category for {same_how:.1%} of {len(trains)} trips; drawn in both: {drawn.sum()}; "
      f"largest position difference {dist.max():.2f} map units (a station circle is 8.2)")
assert same_how == 1.0 and dist.max() < 1.0, "the export is missing something locate() needs"
print("✅ The export alone reproduces §6.")

---
## 9. Next steps

| Step | What it involves | Builds on |
|---|---|---|
| **Swift screensaver** | A preview app + `.saver` sharing one codebase: draw `map_black.pdf` once, download the feeds (SwiftProtobuf), port `locate()`, animate dots with Core Animation. Milestones: map on screen → live dots + measured CPU → installable `.saver` → polish | §8 export, `locate_from_export` |
| **Smooth motion** | Poll every 30 s and remember each trip's last position, so dots glide instead of jumping: never move backwards, wait at a station rather than jump back if a train runs late, catch up gently if it's early. Memory also solves the Rockaway branch ambiguity. | `locate`, `hop_fraction`, `omap.hop_line` |
| **Next train panel** | For your chosen station (all platforms in its complex): next 2–3 trains per direction, destination and minutes, grayed out if you can't make it given your walk time | `complex_of`, `trains["upcoming"]` |
| **Service status** | The MTA alerts feed (`camsys%2Fsubway-alerts.json`, no API key) for delays and planned work | new |
| **Focused mode** | Let the screensaver show a focused view (or rotate through several) instead of the whole map | §7, `FOCUS_VIEWS` |

Known rough edge: a minority of moving trains use straight lines between stations, because some route lines are split into separate pieces in the PDF.

---
# Appendix A. A generated diagram (shareable)

The official map can't be shared without MTA's permission. This appendix builds a diagram in the same spirit purely from GTFS data, so it's yours to share and updates itself when service changes. The main pipeline doesn't depend on it.

**Deterministic by design:** Python visits sets of text IDs in a different order in every session, so wherever this code walks a set or a station pair it sorts first. Without that, routes that tie (e.g. N and W) could swap sides between runs.

What makes it a "new MTA map"-style diagram:
* every line runs horizontal, vertical or at 45°;
* Manhattan stands upright (on a true-north map it leans ~30°) and is enlarged, while outer boroughs shrink;
* each route is its own colored line, with a dot only where *that* route stops.

| Step | What happens |
|---|---|
| A1 | **Trunk graph**: one edge per stretch of track between neighboring stations; express hops folded onto the local track they run beside |
| A2 | **Rotate and distort** the geography: Manhattan upright and wider, a magnifying "fisheye" on the densest area |
| A3 | **Solve the layout**: station positions that make edges 0°/45°/90° while staying near the distorted geography |
| A4 | **One line per route**, offset side by side, plus the line each train follows between stops |
| A5 | Draw it |

**Uses:** `segments`, `patterns`, `stations`, `stop_xy`, `parent_of`, `station_name`, `transfers`, `COLOR`
**Produces:** `diag_xy` (station positions) · `route_lines` · `hop_lines` · `route_station_xy` · `draw_diagram()`

In [ ]:
# ---- Diagram settings (tune these, then re-run Appendix A) ----
STRETCH = 1.8  # how much wider the Manhattan band gets (1 = true width)
BAND_HALF_WIDTH_FT = 7000  # half-width of the Manhattan band, measured from the 6 Av line
FISHEYE = 2.0  # magnification at the fisheye center is FISHEYE + 1 (0 = no fisheye)
FISHEYE_RADIUS_MI = 8.0  # beyond this distance from the center nothing is magnified
LENGTH_POWER = 0.5  # edge length target: 1 = geographic lengths, 0 = every edge equally long
D_MIN = 1.0  # minimum distance between unrelated stations (1 = a typical edge)
CHAIN_TOL = 0.6  # how far a run of track may wiggle before it gets a bend (higher = longer straight runs)
ITERATIONS = 15  # solve → find overlaps → solve again, this many times
W_GEO, W_DIR, W_LEN, W_SMOOTH, W_LINK, W_REPEL = 0.05, 25.0, 1.0, 0.3, 5.0, 3.0  # spring stiffnesses (A3)
SPACING = 0.11  # gap between parallel route lines, in typical-edge units
STRAIGHT_DEG = 6.0  # edges within this many degrees of 0/45/90 are drawn as one straight segment
ROUTE_ORDER = ["1", "2", "3", "4", "5", "6", "6X", "7", "7X", "A", "C", "E", "B", "D", "F", "FX", "M",
               "G", "J", "Z", "L", "N", "Q", "R", "W", "GS", "FS", "H", "SI"]  # tie-break order for parallel lines

### A1. The trunk graph

Segments are per *platform* and per *pattern*, so the 2/3 express from 96 St to 72 St is its own segment lying right on top of the 1 local's. A diagram draws that as **one corridor**, so we:

1. switch from platforms to stations and merge both directions into one edge;
2. find **express hops**: an edge whose two stations are also connected by a path through other stations that is at most 30% longer *and* hugs the express track (every intermediate station within 800 ft of it). The express edge is removed and its routes are added to the local path's edges.

In [ ]:
station_xy = {s: np.array(stop_xy[s]) for s in stations["stop_id"]}  # station → (x, y) feet as an array

trunk = {}  # frozenset({station_a, station_b}) → {"routes": set, "geom": one representative track piece}
for (a, b), seg in segments.items():
    pa, pb = parent_of[a], parent_of[b]
    if pa != pb:
        e = trunk.setdefault(frozenset((pa, pb)), {"routes": set(), "geom": seg["geom"]})
        e["routes"] |= seg["routes"]

G = nx.Graph()  # the trunk graph: nodes = stations, edges = track between neighboring stations
for e, d in trunk.items():
    G.add_edge(*sorted(e), length=d["geom"].length, routes=set(d["routes"]))  # sorted: same order every run

express = {}  # (a, b) → the stations an express hop from a to b runs past (both directions stored)
for e in sorted(trunk, key=lambda e: (-trunk[e]["geom"].length, sorted(e))):  # longest hops first
    u, v = sorted(e)
    data = G.edges[u, v]
    G.remove_edge(u, v)  # temporarily, to look for a local path around it
    try:
        path = nx.shortest_path(G, u, v, weight="length")
    except nx.NetworkXNoPath:
        path = None
    if path is not None:
        path_len = sum(G.edges[a, b]["length"] for a, b in zip(path, path[1:]))
        hugs_track = all(Point(station_xy[n]).distance(trunk[e]["geom"]) < 800 for n in path[1:-1])
    if path is not None and path_len < 1.3 * data["length"] and hugs_track:
        express[(u, v)], express[(v, u)] = path, path[::-1]
        for a, b in zip(path, path[1:]):
            G.edges[a, b]["routes"] |= data["routes"]
    else:
        G.add_edge(u, v, **data)  # not an express hop: put the edge back


def expand(path):
    """Replace express hops in a list of stations with the local stations they run past (recursively)."""
    out = [path[0]]
    for a, b in zip(path, path[1:]):
        out += [b] if G.has_edge(a, b) else expand(express[(a, b)])[1:]
    return out


print(f"Trunk graph: {G.number_of_nodes()} stations, {G.number_of_edges()} edges; {len(express) // 2} express hops merged")
print("e.g. the 2/3 express from 96 St to 72 St runs past:", " → ".join(station_name[n] for n in express[("120", "123")]))

### A2. Rotate and distort

* **Rotate.** Measure Manhattan's lean from the Lexington Av stations (125 St → 14 St): their best-fit direction (first principal component, via SVD). Rotate everything counter-clockwise by that angle.
* **Stretch.** Widen everything within `BAND_HALF_WIDTH_FT` of 6 Av by `STRETCH`, and shift everything outside the band so nothing overlaps.
* **Fisheye.** Magnify around Canal St, the densest area: points near the center spread out, points at `FISHEYE_RADIUS_MI` stay put.

In [ ]:
lex = np.array([station_xy[f"6{n}"] for n in range(21, 36)])  # Lexington Av: 125 St (621) → 14 St (635)
axis = np.linalg.svd(lex - lex.mean(axis=0))[2][0]  # unit vector along the best-fit line
MANHATTAN_ANGLE = np.arctan2(abs(axis[0]), abs(axis[1]))  # radians east of true north
print(f"Manhattan's avenues run {np.degrees(MANHATTAN_ANGLE):.1f}° east of true north")

cos_a, sin_a = np.cos(MANHATTAN_ANGLE), np.sin(MANHATTAN_ANGLE)
ROTATE = np.array([[cos_a, -sin_a], [sin_a, cos_a]])  # counter-clockwise rotation matrix


def rotate(p):
    """Rotate points (N×2 array, or one point) so Manhattan's avenues point straight up."""
    return p @ ROTATE.T


x_mid = rotate(station_xy["D17"])[0]  # center of the Manhattan band: 34 St-Herald Sq on the 6 Av line


def stretch_manhattan(p):
    """Widen the Manhattan band by STRETCH and shift everything outside it outward (N×2 array → N×2)."""
    p = np.array(p, dtype=float)
    dx = p[:, 0] - x_mid
    inside = np.abs(dx) <= BAND_HALF_WIDTH_FT
    outside_shift = np.sign(dx) * (STRETCH * BAND_HALF_WIDTH_FT + np.abs(dx) - BAND_HALF_WIDTH_FT)
    p[:, 0] = x_mid + np.where(inside, STRETCH * dx, outside_shift)
    return p


fisheye_center = stretch_manhattan(rotate(station_xy["639"])[None, :])[0]  # Canal St (6 line)


def fisheye(p):
    """Magnify around `fisheye_center` (N×2 → N×2).

    Within the radius, a point at normalized distance r moves to (F+1)·r / (F·r + 1): r=0 stays put,
    r=1 stays put, and everything in between is pushed outward (strongest near the center).
    """
    v = p - fisheye_center
    r = np.linalg.norm(v, axis=1, keepdims=True) / (FISHEYE_RADIUS_MI * 5280)
    r_new = np.where(r < 1, (FISHEYE + 1) * r / (FISHEYE * r + 1), r)
    return fisheye_center + v * np.where(r > 0, r_new / np.maximum(r, 1e-12), 1.0)


def distort(p):
    """Full geography → diagram warp: rotate, then stretch Manhattan, then fisheye."""
    return fisheye(stretch_manhattan(rotate(p)))


nodes = list(G.nodes)  # fixed station order: row i of every position array is nodes[i]
node_index = {n: i for i, n in enumerate(nodes)}
geo_xy = np.array([station_xy[n] for n in nodes])

fig, axes = plt.subplots(1, 3, figsize=(18, 7), facecolor="black")
for ax, (title, P) in zip(axes, {"True geography": geo_xy, "Rotated": rotate(geo_xy),
                                  "Rotated + stretched + fisheye": distort(geo_xy)}.items()):
    ax.set_facecolor("black")
    for u, v, d in G.edges(data=True):
        a, b = P[node_index[u]], P[node_index[v]]
        ax.plot([a[0], b[0]], [a[1], b[1]], color=COLOR.get(min(d["routes"]), "#888"), lw=0.8)
    ax.set_title(title, color="white")
    ax.set_aspect("equal")
    ax.axis("off")
plt.show()

### A3. Solve the layout: a system of springs

We want a position for each station that satisfies many wishes at once. Each wish is a *linear* equation in the unknown coordinates, so "satisfy them all as well as possible" is **weighted least squares** (the maths of a regression fit), solved with `scipy.sparse.linalg.lsqr`. Think of each equation as a spring whose weight is its stiffness.

| Spring | Equation (≈ true) | Weight |
|---|---|---|
| Stay near your distorted-geography spot | `p_i = anchor_i` | `W_GEO` (weak) |
| Each edge points in its assigned 0°/45°/90° direction **d** | `n · (p_b − p_a) = 0`, **n** ⟂ **d** | `W_DIR` (very strong) |
| Each edge has its target length | `d · (p_b − p_a) = target` | `W_LEN` |
| Stations along a line are evenly spaced | `p_i = (p_prev + p_next) / 2` | `W_SMOOTH` (weak) |
| Stations in one complex stay together | `p_b − p_a = small offset` | `W_LINK` (strong) |
| Unrelated stations don't overlap | `u · (p_j − p_i) = D_MIN` | `W_REPEL` (strong) |

**Directions.** Snapping each edge on its own would make a line at 20° zigzag. Instead the network is split into **chains** (runs of track between junctions); each chain is simplified with Douglas–Peucker into a few straight pieces, and every edge in a piece gets that piece's snapped direction.

The solve repeats `ITERATIONS` times: assign directions → solve → find stations closer than `D_MIN` → add "keep apart" springs.

In [ ]:
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import lsqr
from scipy.spatial import cKDTree

DIRECTIONS = np.array([[np.cos(k * np.pi / 4), np.sin(k * np.pi / 4)] for k in range(8)])  # 0°, 45°, … 315°


def snap_direction(v):
    """The unit vector among the 8 octilinear directions closest to vector v."""
    return DIRECTIONS[np.argmax(DIRECTIONS @ (v / np.linalg.norm(v)))]


# Anchors: distorted geography, scaled so a typical (median) edge is 1 unit long.
anchor = distort(geo_xy)
edge_len = {(u, v): np.linalg.norm(anchor[node_index[v]] - anchor[node_index[u]]) for u, v in G.edges}
unit = np.median(list(edge_len.values()))
anchor = (anchor - anchor.mean(axis=0)) / unit
target_len = {e: max(0.6, (L / unit) ** LENGTH_POWER) for e, L in edge_len.items()}

complex_links = {  # pairs of stations joined by a transfer, both present in the trunk graph
    frozenset((a, b)) for a, b in zip(transfers["from_stop_id"], transfers["to_stop_id"])
    if a != b and a in node_index and b in node_index
}


def find_chains():
    """Split the trunk graph into chains: runs of stations between junctions or terminals (degree ≠ 2)."""
    ends = {n for n in G.nodes if G.degree(n) != 2}
    seen, chains = set(), []
    for start in sorted(ends):  # sorted: chains (and their orientation) come out the same every run
        for first in sorted(G.neighbors(start)):
            if frozenset((start, first)) in seen:
                continue
            chain = [start, first]
            while chain[-1] not in ends:  # walk until the next junction/terminal
                chain.append(next(m for m in G.neighbors(chain[-1]) if m != chain[-2]))
            seen |= {frozenset(h) for h in zip(chain, chain[1:])}
            chains.append(chain)
    return chains


def douglas_peucker(P, tol):
    """Indices of the points of polyline P (N×2) to keep when simplifying with tolerance `tol`."""
    keep, stack = {0, len(P) - 1}, [(0, len(P) - 1)]
    while stack:
        i, j = stack.pop()
        if j <= i + 1:
            continue
        ab = P[j] - P[i]
        rel = P[i + 1:j] - P[i]
        dist = np.abs(ab[0] * rel[:, 1] - ab[1] * rel[:, 0]) / max(np.linalg.norm(ab), 1e-9)  # to line i–j
        k = int(np.argmax(dist))
        if dist[k] > tol:  # farthest point is too far: keep it and split there
            keep.add(i + 1 + k)
            stack += [(i, i + 1 + k), (i + 1 + k, j)]
    return sorted(keep)


chains = find_chains()


def chain_directions(pos):
    """Assign each edge an octilinear direction from its chain's simplified shape.

    Args:
        pos: current positions (len(nodes)×2).
    Returns:
        {(a, b): unit direction from a to b} with (a, b) oriented along the chain.
    """
    out = {}
    for chain in chains:
        pts = np.array([pos[node_index[n]] for n in chain])
        breaks = douglas_peucker(pts, CHAIN_TOL)
        for i, j in zip(breaks, breaks[1:]):
            d = snap_direction(pts[j] - pts[i])
            for k in range(i, j):
                out[(chain[k], chain[k + 1])] = d
    return out


class Springs:
    """Collects weighted linear equations  Σ coef·unknown = rhs  and solves them by least squares."""

    def __init__(self):
        self.rows, self.cols, self.vals, self.rhs = [], [], [], []

    def add(self, terms, rhs, weight):
        """Add one equation. `terms` is [(column, coefficient), ...]; both sides are multiplied by `weight`."""
        r = len(self.rhs)
        for col, coef in terms:
            self.rows.append(r)
            self.cols.append(col)
            self.vals.append(weight * coef)
        self.rhs.append(weight * rhs)

    def solve(self, x0):
        """Least-squares solution, starting from x0 (flattened [x0, y0, x1, y1, ...])."""
        A = coo_matrix((self.vals, (self.rows, self.cols)), shape=(len(self.rhs), len(x0))).tocsr()
        return lsqr(A, np.array(self.rhs), x0=x0, atol=1e-10, btol=1e-10, iter_lim=20000)[0]


def col_x(n):
    """Column of station n's x coordinate in the unknowns vector."""
    return 2 * node_index[n]


def col_y(n):
    """Column of station n's y coordinate in the unknowns vector."""
    return 2 * node_index[n] + 1


def solve_layout():
    """Run the spring solve ITERATIONS times; returns final positions (len(nodes)×2)."""
    pos = anchor.copy()
    repel = {}  # (a, b) → unit vector from a to b: pairs to keep at least D_MIN apart (grows each round)
    graph_neighbors = {frozenset(e) for e in G.edges} | complex_links  # these pairs may be close
    for _ in range(ITERATIONS):
        for i, j in cKDTree(pos).query_pairs(D_MIN):
            if frozenset((nodes[i], nodes[j])) not in graph_neighbors:
                d = pos[j] - pos[i]
                repel[(nodes[i], nodes[j])] = d / max(np.linalg.norm(d), 1e-9)

        s = Springs()
        for n in nodes:  # stay near the anchor
            s.add([(col_x(n), 1)], anchor[node_index[n], 0], W_GEO)
            s.add([(col_y(n), 1)], anchor[node_index[n], 1], W_GEO)
        for (a, b), d in chain_directions(pos).items():  # direction and length of every edge
            nrm = np.array([-d[1], d[0]])
            s.add([(col_x(b), nrm[0]), (col_y(b), nrm[1]), (col_x(a), -nrm[0]), (col_y(a), -nrm[1])], 0.0, W_DIR)
            L = target_len.get((a, b), target_len.get((b, a)))
            s.add([(col_x(b), d[0]), (col_y(b), d[1]), (col_x(a), -d[0]), (col_y(a), -d[1])], L, W_LEN)
        for n in nodes:  # even spacing along lines
            nb = list(G.neighbors(n))
            if len(nb) == 2:
                for col in (col_x, col_y):
                    s.add([(col(n), 1), (col(nb[0]), -0.5), (col(nb[1]), -0.5)], 0.0, W_SMOOTH)
        for link in sorted(complex_links, key=sorted):  # complexes stay together, in their real relative direction
            a, b = sorted(link)
            off = anchor[node_index[b]] - anchor[node_index[a]]
            off = 0.35 * off / max(np.linalg.norm(off), 1e-9)
            s.add([(col_x(b), 1), (col_x(a), -1)], off[0], W_LINK)
            s.add([(col_y(b), 1), (col_y(a), -1)], off[1], W_LINK)
        for (a, b), u in repel.items():  # keep apart
            s.add([(col_x(b), u[0]), (col_y(b), u[1]), (col_x(a), -u[0]), (col_y(a), -u[1])], D_MIN, W_REPEL)

        pos = s.solve(pos.ravel()).reshape(-1, 2)
    return pos


t0 = time.time()
layout = solve_layout()
diag_xy = {n: layout[node_index[n]] for n in nodes}  # station → diagram position

angle_err = []  # how far each edge ended up from a perfect 0°/45°/90°
for u, v in G.edges:
    w = diag_xy[v] - diag_xy[u]
    angle_err.append(np.degrees(np.arccos(min(1.0, snap_direction(w) @ w / np.linalg.norm(w)))))
print(f"Solved in {time.time() - t0:.1f}s. {len(chains)} chains. "
      f"Edge angle error: mean {np.mean(angle_err):.2f}°, worst {np.max(angle_err):.1f}°")

### A4. One line per route

Each route is drawn as its own line, offset sideways from the corridor center:

1. **Which side?** For each corridor edge, each route's nearby stations (6 stops either way) show which side it peels off to; it goes on that side. Ties (routes that run together, like N and W) use `ROUTE_ORDER`.
2. **Exactly octilinear.** Edges within `STRAIGHT_DEG` of 0/45/90° are drawn straight; others as straight–diagonal–straight (a dogleg).
3. **Clean corners.** Offset corners use a *miter* join (intersection of the two offset lines), unless that point would shoot far away on a shallow corner (*miter limit*), in which case a bevel is used.

`hop_lines[(route, a, b)]` is the exact line a train on that route follows from station a to b.

In [ ]:
def unit_vec(v):
    """v scaled to length 1."""
    return v / np.linalg.norm(v)


def left_of(t):
    """The unit normal 90° counter-clockwise (to the left) of direction t."""
    return np.array([-t[1], t[0]])


# Each route's paths through the trunk graph (express hops expanded), always written "northbound",
# with the set of stations where the route actually stops.
route_paths = defaultdict(dict)  # route → {tuple of stations along the path: set of stations it stops at}
for p in patterns.itertuples():
    seq = [parent_of[s] for s in p.stops]
    if p.stops[0].endswith("S"):
        seq = seq[::-1]
    route_paths[p.route_id].setdefault(tuple(expand(seq)), set(seq))

# Orient every edge along its chain so "left" means the same side all along a run of track.
orient = {frozenset(h): h for chain in chains for h in zip(chain, chain[1:])}

edge_routes = defaultdict(set)  # edge → routes running along it
for r, paths in route_paths.items():
    for path in paths:
        for h in zip(path, path[1:]):
            edge_routes[frozenset(h)].add(r)


def lateral_key(r, a, b, context=6):
    """Average signed distance (left of a→b) of route r's stations within `context` stops of edge a–b.

    Negative = the route lies to the right; used to order parallel routes on the edge.
    """
    mid, n = (diag_xy[a] + diag_xy[b]) / 2, left_of(unit_vec(diag_xy[b] - diag_xy[a]))
    vals = []
    for path in route_paths[r]:
        for i in range(len(path) - 1):
            if {path[i], path[i + 1]} == {a, b}:
                vals += [n @ (diag_xy[c] - mid) for c in path[max(0, i - context): i + context + 2]]
    return round(float(np.mean(vals)), 2) if vals else 0.0


slot = {}  # (edge, route) → sideways offset, measured to the left of the edge's chain orientation
for e, routes_here in edge_routes.items():
    a, b = orient[e]
    order = sorted(routes_here, key=lambda r: (lateral_key(r, a, b), ROUTE_ORDER.index(r) if r in ROUTE_ORDER else 99, r))
    for j, r in enumerate(order):
        slot[(e, r)] = (j - (len(order) - 1) / 2) * SPACING


def slot_offset(r, a, b):
    """Route r's sideways offset to the left of the travel direction a→b."""
    e = frozenset((a, b))
    return slot[(e, r)] if orient[e] == (a, b) else -slot[(e, r)]


def edge_points(a, b):
    """Corridor centerline from station a to b, made only of 0°/45°/90° pieces."""
    p, q = diag_xy[a], diag_xy[b]
    w = q - p
    d1 = snap_direction(w)
    if d1 @ w / np.linalg.norm(w) > np.cos(np.radians(STRAIGHT_DEG)):
        return [p, q]
    turn = d1[0] * w[1] - d1[1] * w[0]  # which side of d1 the target lies on
    k = int(np.argmax(DIRECTIONS @ d1))
    d2 = DIRECTIONS[(k + (1 if turn > 0 else -1)) % 8]  # the neighboring 45° direction on that side
    alpha, beta = np.linalg.solve(np.column_stack([d1, d2]), w)  # w = alpha·d1 + beta·d2
    m1 = p + d1 * alpha / 2
    return [p, m1, m1 + d2 * beta, q]  # half straight, diagonal, half straight


def offset_path(r, path):
    """Route r's own (offset) line along a list of stations.

    Returns:
        (points, station_at): the polyline as an N×2 array, and for each station in `path` the index of
        its point in that array.
    """
    legs, first_leg = [], []  # legs: (start, end, offset) for each straight piece
    for a, b in zip(path, path[1:]):
        o = slot_offset(r, a, b)
        pts = edge_points(a, b)
        first_leg.append(len(legs))
        legs += [(p0, p1, o) for p0, p1 in zip(pts, pts[1:]) if np.linalg.norm(p1 - p0) > 1e-9]
    hop_starts = set(first_leg[1:])  # leg indices where a new station begins
    out, station_at = [], [0]
    for i, (p0, p1, o) in enumerate(legs):
        t = unit_vec(p1 - p0)
        n = left_of(t)
        if i > 0:
            q0, q1, oq = legs[i - 1]
            tq = unit_vec(q1 - q0)
            nq = left_of(tq)
            turn = tq[0] * t[1] - tq[1] * t[0]
            miter = None
            if abs(turn) > 1e-3:  # a real corner: intersect the two offset lines
                s_ = np.linalg.solve(np.column_stack([tq, -t]), (p0 + o * n) - (q0 + oq * nq))[0]
                miter = q0 + oq * nq + s_ * tq
                if np.linalg.norm(miter - p0) > 3 * max(abs(o), abs(oq), SPACING):
                    miter = None  # miter limit: too far away, bevel instead
            if miter is not None:
                out[-1] = miter
            elif abs(o - oq) > 1e-9 or abs(turn) > 1e-3:
                out.append(p0 + o * n)  # bevel, or a small jog where the number of parallel routes changes
            if i in hop_starts:
                station_at.append(len(out) - 1)
        else:
            out.append(p0 + o * n)
        out.append(p1 + o * n)
    station_at.append(len(out) - 1)
    return np.array(out), station_at


route_lines = []  # (route, N×2 points) for drawing
stop_dots = []  # (x, y) wherever a route stops
hop_lines = {}  # (route, from_station, to_station) → LineString a train follows
route_station_xy = {}  # (route, station) → that route's point at the station

for r, paths in route_paths.items():
    for path, stops_here in paths.items():
        pts, station_at = offset_path(r, list(path))
        assert len(station_at) == len(path)
        route_lines.append((r, pts))
        stop_idx = [i for i, n in enumerate(path) if n in stops_here]
        for i in stop_idx:
            route_station_xy.setdefault((r, path[i]), pts[station_at[i]])
            stop_dots.append(pts[station_at[i]])
        for i, j in zip(stop_idx, stop_idx[1:]):
            piece = pts[station_at[i]: station_at[j] + 1]
            if len(piece) >= 2:
                hop_lines.setdefault((r, path[i], path[j]), LineString(piece))
                hop_lines.setdefault((r, path[j], path[i]), LineString(piece[::-1]))

stop_dots = np.array(stop_dots)
print(f"{len(route_lines)} route lines, {len(hop_lines) // 2} stop-to-stop hops, {len(stop_dots)} station dots")

### A5. The generated diagram

Known rough spots: downtown Manhattan (Fulton St / Chambers St) is crowded, and a few routes swap sides where bundles merge. A low-tech fix would be pinning a few stations by hand with a very strong anchor spring.

In [ ]:
def draw_diagram(ax, lw=1.2, dot_size=3, link_lw=2.5):
    """Draw the generated diagram on `ax`: transfer connectors, route lines, then stop dots."""
    ax.set_facecolor("black")
    for link in complex_links:
        a, b = sorted(link)
        ax.plot(*np.array([diag_xy[a], diag_xy[b]]).T, color="#bbbbbb", lw=link_lw, solid_capstyle="round", zorder=1)
    for r, pts in route_lines:
        ax.plot(pts[:, 0], pts[:, 1], color=COLOR.get(r, "#888888"), lw=lw, solid_joinstyle="miter", zorder=2)
    ax.scatter(stop_dots[:, 0], stop_dots[:, 1], s=dot_size, c="white", linewidths=0, zorder=3)
    ax.set_aspect("equal")
    ax.axis("off")


def zoom_box(station_ids, pad=1.0):
    """Axis limits (x0, x1, y0, y1) around the given stations' diagram positions."""
    pts = np.array([diag_xy[s] for s in station_ids])
    (x0, y0), (x1, y1) = pts.min(axis=0) - pad, pts.max(axis=0) + pad
    return x0, x1, y0, y1


fig, (ax_all, ax_zoom) = plt.subplots(1, 2, figsize=(18, 12), facecolor="black", gridspec_kw={"width_ratios": [1.1, 1]})
draw_diagram(ax_all, lw=1.0, dot_size=2, link_lw=2)
draw_diagram(ax_zoom, lw=2.4, dot_size=12, link_lw=5)
x0, x1, y0, y1 = zoom_box(["120", "621", "A27", "D17", "629", "639", "A41", "R31"])  # 96 St … Atlantic Av
ax_zoom.set_xlim(x0, x1), ax_zoom.set_ylim(y0, y1)
plt.tight_layout()
plt.show()

---
# Appendix B. Explorations

Questions answered along the way. None of this is needed by the pipeline; it's here for reference and for re-checking when the data changes.

### B1. Which station is served by the most routes?

**Answer:** as single GTFS stations, **DeKalb Av** (6 routes). But GTFS splits big hubs into one station per original line (built by three rival companies: IRT, BMT, IND). Merged into complexes, **Times Sq–42 St / Port Authority** wins: 13 routes over 5 GTFS stations.

In [ ]:
served = (  # (station, route) pairs from the schedule
    trip_stops[["stop_id", "route_id"]].assign(station=lambda d: d["stop_id"].map(parent_of))
    .drop_duplicates(["station", "route_id"])
)
by_station = served.groupby("station")["route_id"].agg(lambda r: " ".join(sorted(r))).to_frame("routes")
by_station["n"] = by_station["routes"].str.split().str.len()
by_station["name"] = by_station.index.map(station_name)
print(by_station.sort_values("n", ascending=False).head(5).to_string())

print()
by_complex = served.assign(complex=served["station"].map(complex_of)).groupby("complex")
for cid, grp in sorted(by_complex, key=lambda kv: -kv[1]["route_id"].nunique())[:4]:
    member_names = " / ".join(sorted({station_name[s] for s in grp["station"]}))
    print(f"{grp['route_id'].nunique():2d} routes, {grp['station'].nunique()} GTFS stations: {member_names}")

### B2. How much track simplification is invisible?

The MacBook screen is 1964 pixels tall. If the whole system (≈ 27 mi) filled it, one pixel ≈ 72 ft, so simplifying by up to about half a pixel (~36 ft) can't be seen. `SIMPLIFY_TOL_FT = 25` is safe. This only affects the true-geography map; both diagrams draw straight lines between stations.

In [ ]:
height_ft = stops["y"].max() - stops["y"].min()
ft_per_px = height_ft / 1964
print(f"One pixel ≈ {ft_per_px:.0f} ft → up to ~{ft_per_px / 2:.0f} ft of simplification is invisible full-screen")

fig, axes = plt.subplots(1, 4, figsize=(20, 5), facecolor="black")
xmin, xmax, ymin, ymax = bbox_around(40.6906, -73.9818, 4000)  # around DeKalb Av
for ax, tol in zip(axes, [0, 25, 100, 400]):
    n_pts = 0
    ax.set_facecolor("black")
    for seg in segments.values():
        line = seg["geom"].simplify(tol)
        n_pts += len(line.coords)
        ax.plot(*line.xy, color=COLOR.get(min(seg["routes"]), "#888888"), lw=1.5)
    ax.set_xlim(xmin, xmax), ax.set_ylim(ymin, ymax), ax.set_aspect("equal"), ax.axis("off")
    ax.set_title(f"tolerance {tol} ft: {n_pts:,} points", color="white")
plt.show()

### B3. Trips per line right now

Changes a lot with the time of day: late at night some lines (B, W, the express variants) disappear. The counts include trips that haven't started yet.

In [ ]:
counts = trains["route_id"].value_counts()
fig, ax = plt.subplots(figsize=(12, 3), facecolor="black")
ax.set_facecolor("black")
ax.bar(counts.index, counts.values, color=[COLOR.get(r, "#888888") for r in counts.index])
ax.tick_params(colors="white")
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title(f"Trips in the feed at {time.strftime('%-I:%M %p', time.localtime(fetched_at))}: "
             f"most on the {counts.index[0]} ({counts.iloc[0]})", color="white")
plt.show()

### B4. Does pattern matching beat "most common previous stop"?

`previous_stop()` (§6) picks the pattern that best matches a train's upcoming stops. The simpler alternative is the *most common* previous stop. They only disagree when service differs from the usual pattern, mostly late at night, so the live count is often 0 during the day. The made-up late-night local 2 train shows the difference any time.

In [ ]:
def most_common_previous(route, stop):
    """The platform that most often precedes `stop` on `route`, by trips per day (the naive guess)."""
    c = Counter()
    for stops_, n in route_patterns.get(route, []):
        if stop in stops_ and stops_.index(stop) > 0:
            c[stops_[stops_.index(stop) - 1]] += n
    return c.most_common(1)[0][0] if c else None


moving = trains[trains["how"].str.startswith("between")]
disagree = []
for _, t in moving.iterrows():
    smart = previous_stop(t["route_id"], t["vehicle_stop"], [s for s, _ in t["upcoming"]])
    naive = most_common_previous(t["route_id"], t["vehicle_stop"])
    if naive and smart != naive:
        disagree.append((t["route_id"], t["vehicle_stop"], naive, smart))
print(f"{len(disagree)} of {len(moving)} moving trains would be on a different hop with the naive guess")

local_2 = ["132N", "131N", "130N"]  # a northbound 2 heading to 14 St, then 18 St, 23 St (running local)
print(f"Made-up local 2 train at 14 St: naive guess = {station_name[parent_of[most_common_previous('2', '132N')]]}, "
      f"pattern match = {station_name[parent_of[previous_stop('2', '132N', local_2)]]}")
pd.DataFrame(disagree, columns=["route", "heading to", "naive guess", "pattern match"]).head(10)